# 04. Market Analyst Agent (`MarketAnalyst`) Demo

### Architecture Role & Responsibilities
The **Market Analyst** examines price action, technical momentum, and macroeconomic backdrop. It computes technical indicators using its integrated tool:

- `get_market_technicals`: 50-day and 200-day SMAs, 14-day RSI, 52-week range, and beta.

In [1]:
import sys
from pathlib import Path

# Resolve src directory
project_root = Path.cwd().parent if Path.cwd().name == "demo" else Path.cwd()
sys.path.insert(0, str(project_root / "src"))

from research_agent.agents.market_analyst import market_analyst_agent
from research_agent.interfaces import get_llm

llm = get_llm()
market_analyst = market_analyst_agent(llm=llm)
print(f"Agent: {market_analyst.name}")
print(f"Tools: {list(market_analyst.tools.keys())}")

Agent: MarketAnalyst
Tools: ['get_market_technicals', 'get_macro_indicators']


## Sending Technical Analysis Request

In [2]:
prompt = "Evaluate market price momentum, SMAs, RSI, and technical indicators for NVDA"
print(f"Sending Input: '{prompt}'\n")

resp = market_analyst.run(prompt)
print("--- MARKET ANALYST OUTPUT ---")
print(resp.content)

Sending Input: 'Evaluate market price momentum, SMAs, RSI, and technical indicators for NVDA'

--- MARKET ANALYST OUTPUT ---
**NVDA – Technical Snapshot (as of 2026‑10‑12)**  

| Metric | Value |
|--------|-------|
| **Current price** | **$229.28** |
| **50‑day SMA** | **$221.94** |
| **200‑day SMA** | **$201.83** |
| **14‑day RSI** | **53.0** |
| **Beta (1‑yr)** | **2.22** |
| **52‑week range** | **Low $164.27 – High $243.37** |
| **Trend label (tool‑generated)** | **Bullish Uptrend** |

---

### 1. Trend Structure  
- The price sits **~7 % above the 50‑day SMA** and **~14 % above the 200‑day SMA**, confirming the bullish trend indicated by the tool’s “Bullish Uptrend” label.  
- Being above both moving averages implies that short‑term momentum is aligned with the longer‑term bullish bias.

### 2. Momentum Oscillator (RSI)  
- RSI‑14 = **53**, comfortably in the neutral band (30‑70).  
- No over‑bought (>70) or over‑sold (<30) condition is present, suggesting there is still room for p

## Inspecting Tool Execution Trace

In [3]:
print("--- TOOL EXECUTION TRACE ---")
for step in resp.trace:
    if step.step_type == "tool_call":
        print(f"TOOL CALL   : {step.content.get('tool')}({step.content.get('arguments')})")
    elif step.step_type == "tool_result":
        preview = str(step.content)[:90].replace("\n", " ")
        print(f"OBSERVATION : {preview}...")
    elif step.step_type == "final_answer":
        print(f"FINAL ANSWER: Generated ({len(str(step.content))} chars)")

--- TOOL EXECUTION TRACE ---
TOOL CALL   : get_market_technicals({'ticker': 'NVDA'})
OBSERVATION : {   "ticker": "NVDA",   "current_price": 229.28,   "52_week_high": 243.37,   "52_week_low"...
FINAL ANSWER: Generated (3166 chars)
